## 1. Analise de metadata/schema arquivos parquet

### a) Imports e path

In [0]:
%load_ext autoreload
%autoreload 2

import sys
import os
import pandas as pd
from pyspark.sql.types import StructType, StructField, LongType, DoubleType, TimestampType, StringType

# -- Path -- #
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
print(project_root)

### b) Primeiras linhas (Janeiro-2023)

In [0]:
%load_ext autoreload
%autoreload 2

from src.config.settings import RAW_VOLUME_PATH

january_path = f"{RAW_VOLUME_PATH}/yellow/yellow_tripdata_2023-01.parquet"

print(f"Lendo dados raw do mês 01...\nCaminho: {january_path}")

df_raw_january = spark.read.parquet(january_path)
col_qty = len(df_raw_january.columns)

print(f"Total de registros em Janeiro, Qtd de Linhas: {df_raw_january.count():,} | Qtd de colunas: {col_qty}\n")
print("Exibindo as 10 primeiras linhas:")
display(df_raw_january.limit(10))

### c) Analise datatypes por mes
Objetivo: Mapear schema drift entre as partições mensais. \
O auto-detect do Spark é insuficiente para o upcasting necessário, sendo indispensável a padronização via Schema Enforcement para evitar que colunas com tipos conflitantes (ex: Double vs Long) causem falhas na unificação.

In [0]:
%load_ext autoreload
%autoreload 2

from src.config.settings import RAW_VOLUME_PATH, TARGET_YEARS, TARGET_MONTHS

print("Extraindo metadados dos arquivos dinamicamente...\n")

schema_dict = {}

for year in TARGET_YEARS:
    for month in TARGET_MONTHS:
        formatted_month = str(month).zfill(2)
        database_path = f"{RAW_VOLUME_PATH}/yellow/yellow_tripdata_{year}-{formatted_month}.parquet"
        period_key = f"{year}_{formatted_month}"
        
        try:
            file_schema = spark.read.parquet(database_path).schema
            schema_dict[period_key] = {
                data_field.name: data_field.dataType.simpleString() 
                for data_field in file_schema.fields
            }
        except Exception as e:
            print(f"Erro ao ler o arquivo do período {period_key}. Verifique se o caminho esta correto. Erro: {e}")

df_comparison = pd.DataFrame(schema_dict)
df_comparison = df_comparison.fillna("--- AUSENTE ---")

df_comparison = df_comparison.reset_index()
df_comparison = df_comparison.rename(columns={'index': 'column_name'})

period_columns = [col for col in df_comparison.columns if col != 'column_name']

type_conflicts = df_comparison[df_comparison[period_columns].nunique(axis=1) > 1]

if not type_conflicts.empty:
    print("ATENÇÃO! Foram encontrados conflitos de tipos nas seguintes colunas:")
    display(type_conflicts)
else:
    print("Todos os arquivos têm exatamente o mesmo esquema e tipos de dados!")

print("\n--- Esquema completo de todos os períodos ---")
display(df_comparison)

### d) Teste Schema Enforcement
Conclusão: Para as 5 colunas mapeadas com diferenças, testamos upcasting com schema enforcement. \
O teste foi bem-sucedido.

In [0]:
%load_ext autoreload
%autoreload 2

from src.config.settings import RAW_VOLUME_PATH

bronze_schema = StructType([
    StructField("VendorID", LongType(), True), #<-- Diferença de schema
    StructField("tpep_pickup_datetime", TimestampType(), True), 
    StructField("tpep_dropoff_datetime", TimestampType(), True),
    StructField("passenger_count", LongType(), True), #<-- Diferença de schema
    StructField("trip_distance", DoubleType(), True),
    StructField("RatecodeID", LongType(), True), #<-- Diferença de schema
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", LongType(), True), #<-- Diferença de schema
    StructField("DOLocationID", LongType(), True), #<-- Diferença de schema
    StructField("payment_type", LongType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
    StructField("airport_fee", DoubleType(), True)
])

print("Lendo a base Bronze com Schema Enforcement...")
df_silver = (
    spark.read
    .schema(bronze_schema)
    .parquet(f"{RAW_VOLUME_PATH}/yellow/*.parquet")
)


rows_qty_silver = df_silver.count()
col_qty_silver = len(df_silver.columns)
print(f"Total de registros em Janeiro, Qtd de Linhas: {df_silver.count():,} | Qtd de colunas: {col_qty}\n")
print("\n10 primeiras linhas do DataFrame unificado:")
display(df_silver.limit(10))
conflict_columns = ["VendorID", "passenger_count", "RatecodeID", "PULocationID", "DOLocationID"]
print("\nColunas que possuíam conflitos de tipo:")
display(df_silver.select(*conflict_columns).limit(10))

